##  Experiments

In [1]:
import json
import numpy as np
import pandas as pd
import os
from google.cloud import bigquery
from google.api_core.exceptions import NotFound
from google.cloud import storage
import joblib
from io import BytesIO
from dotenv import load_dotenv
import types
import sys

from datetime import datetime
import pytz
from uuid import uuid4

pd.set_option("display.max_columns", None)

# Definiendo ruta de ejecucion del proyecto
PATH_ROOT = os.getcwd()
post_project = [i for i, val in enumerate(str(PATH_ROOT).split('\\')) if val == 'project-data-processing'][0]
os.chdir('\\'.join(str(PATH_ROOT).split('\\')[:post_project + 1]))

d:\03_PROYECTOS\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages\google\api_core\_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.25). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
d:\03_PROYECTOS\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages\google\auth\__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
d:\03_PROYECTOS\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages\google\oauth2\_

In [2]:
AAAAMM = '202605'

In [3]:
load_dotenv()

project_id = os.getenv("GCP_PROJECT_ID")
bucket_name = os.getenv("GCP_BUCKET_NAME")
model_ruta = os.getenv("MODEL_ROOT")
pipeline_ruta = os.getenv("PIPELINE_ROOT")
credentials_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# Tablas inputs
project_id_input = os.getenv("GCP_PROJECT_ID_INPUT")
dataset_id_input = os.getenv("BQ_DATASET_ID_INPUT")
table_id_input = os.getenv("BQ_TABLE_ID_INPUT")

# Tablas features
dataset_id_features = os.getenv("BQ_DATASET_ID_FEATURES")
table_id_features = os.getenv("BQ_TABLE_ID_FEATURES")

# Tablas temporales
dataset_temp = os.getenv("BQ_DATASET_ID_TEMP")
table_temp_data_input = os.getenv("BQ_TABLE_ID_TEMP_DATA")
table_temp_data_transf_input = os.getenv("BQ_TABLE_ID_TEMP_DATA_TRANSF")
table_temp_data_predict = os.getenv("BQ_TABLE_ID_TEMP_DATA_PREDICT")

# Tablas Ouputs
project_id_out = os.getenv("GCP_PROJECT_ID_OUT")
dataset_id_out = os.getenv("BQ_DATASET_ID_OUT")
table_out = os.getenv("BQ_TABLE_ID_OUT")
table_out_hist = os.getenv("BQ_TABLE_ID_OUT_HIST")


In [4]:
model_name  = "Model-GradientBoostingRegressor.joblib"
model_path = f"{model_ruta}/{model_name}"
print(model_path)

pipeline_name = 'Pipeline-Transformacion-Training.joblib'
pipeline_path = f"{pipeline_ruta}/{pipeline_name}"
print(pipeline_path)

metadata_name = 'metadata_transformer.py'
metadata_path = f"{pipeline_ruta}/{metadata_name}"
print(metadata_path)

project-pipeline-predictions-casas/models/Model-GradientBoostingRegressor.joblib
project-pipeline-predictions-casas/pipeline/Pipeline-Transformacion-Training.joblib
project-pipeline-predictions-casas/pipeline/metadata_transformer.py


In [5]:
def generar_run_id() -> str:
    timestamp = datetime.now(pytz.utc).strftime("%Y%m%dT%H%M%SZ")
    token = uuid4().hex[:12]
    return f"house-price-{timestamp}-{token}"

run_id = generar_run_id()

## Componente 1

In [6]:
client = bigquery.Client(project=project_id)

path_bq_table = f"{project_id_input}.{dataset_id_input}.{table_id_input}"

dfInput = client.query(
        f'''SELECT * EXCEPT(date_subida_local,date_subida_utc)
        FROM `{path_bq_table}` where periodo = '{AAAAMM}'
        '''
    ).to_dataframe()

path_bq_table_feature = f"{project_id}.{dataset_id_features}.{table_id_features}"
dfFeatures = client.query(
        f'''SELECT * FROM `{path_bq_table_feature}`
        '''
    ).to_dataframe()

listFeatures = dfFeatures['features'].tolist()

dfInputFeat = dfInput[['id','periodo'] + listFeatures].copy()

user_id = client.query("SELECT SESSION_USER()").to_dataframe().iloc[0, 0]
fecha_carga = datetime.now(pytz.timezone("America/Lima"))

dfInputFeat['run_id'] = run_id
dfInputFeat['creation_user'] = user_id
dfInputFeat['str_process_date_local'] = fecha_carga.replace(tzinfo=None).strftime('%Y%m%d')
dfInputFeat['process_datetime_local'] = fecha_carga.replace(tzinfo=None)
dfInputFeat['process_datetime_utc'] = fecha_carga.astimezone(pytz.UTC)


dfInputFeat = dfInputFeat[  ['id','periodo'] 
                          + listFeatures 
                          + ['run_id','creation_user','str_process_date_local','process_datetime_local','process_datetime_utc']]
periodo = dfInputFeat['periodo'].iloc[0]
dfInputFeat

,id,periodo,mssubclass,mszoning,lotfrontage,lotarea,neighborhood,overallqual,overallcond,yearbuilt,yearremodadd,bsmtqual,bsmtfintype_principal,bsmtfinsf_principal,totalbsmtsf,centralair,firstflrsf,secondflrsf,grlivarea,kitchenqual,fireplacequ,garagetype,garagecars,garagearea,yrsold,run_id,creation_user,str_process_date_local,process_datetime_local,process_datetime_utc
0,1461,202605,20,RH,80.0,11622,NAmes,5,6,1961,1961,TA,Rec,468.0,882.0,Y,896.0,0.0,896.0,TA,None,Attchd,1.0,730.0,2010,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:23:48.748524,2026-08-08 19:23:48.748524+00:00
1,1462,202605,20,RL,81.0,14267,NAmes,6,6,1958,1958,TA,ALQ,923.0,1329.0,Y,1329.0,0.0,1329.0,Gd,None,Attchd,1.0,312.0,2010,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:23:48.748524,2026-08-08 19:23:48.748524+00:00
2,1467,202605,20,RL,NaN,7980,Gilbert,6,7,1992,2007,Gd,ALQ,935.0,1168.0,Y,1187.0,0.0,1187.0,TA,None,Attchd,2.0,420.0,2010,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:23:48.748524,2026-08-08 19:23:48.748524+00:00
3,1469,202605,20,RL,85.0,10176,Gilbert,7,5,1990,1990,Gd,GLQ,637.0,1300.0,Y,1341.0,0.0,1341.0,Gd,Po,Attchd,2.0,506.0,2010,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:23:48.748524,2026-08-08 19:23:48.748524+00:00
4,1470,202605,20,RL,70.0,8400,NAmes,4,5,1970,1970,TA,ALQ,804.0,882.0,Y,882.0,0.0,882.0,TA,None,Attchd,2.0,525.0,2010,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:23:48.748524,2026-08-08 19:23:48.748524+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1454,2772,202605,190,RL,70.0,7000,NAmes,5,5,1962,1962,TA,ALQ,953.0,1025.0,Y,1025.0,0.0,1025.0,TA,None,None,0.0,0.0,2006,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:23:48.748524,2026-08-08 19:23:48.748524+00:00
1455,2779,202605,190,RM,56.0,7745,OldTown,4,6,1900,1950,TA,Unf,0.0,938.0,N,1084.0,867.0,1951.0,Fa,None,Detchd,2.0,576.0,2006,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:23:48.748524,2026-08-08 19:23:48.748524+00:00
1456,2784,202605,190,RM,50.0,6000,OldTown,5,7,1955,1955,TA,GLQ,576.0,960.0,Y,960.0,0.0,960.0,TA,None,Detchd,2.0,576.0,2006,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:23:48.748524,2026-08-08 19:23:48.748524+00:00
1457,2875,202605,190,RH,58.0,6430,SWISU,6,6,1945,1950,TA,BLQ,780.0,780.0,N,816.0,524.0,1340.0,TA,None,Attchd,1.0,440.0,2006,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:23:48.748524,2026-08-08 19:23:48.748524+00:00


In [7]:
# Validacion de duplicados: 

dfVAlDup = dfInputFeat.groupby(['id','periodo']).size().reset_index().rename(columns = {0:'cant'})
Res = dfVAlDup[dfVAlDup['cant'] > 1]

if Res.shape[0] > 0:
    print(f"Error de duplicados: {Res.shape[0]} casos")
    print(Res)
    exit()


In [8]:
path_bq_data_input = f"{project_id}.{dataset_temp}.{table_temp_data_input}"

try:

    delete_query = f"""
        DELETE FROM `{path_bq_data_input}`
        WHERE periodo = '{periodo}'
    """

    delete_job = client.query(delete_query)
    delete_job.result()

    print(f"Registros previos eliminados para período {periodo}.")

except NotFound:
    print("La tabla destino no existe aún; se creará durante la carga.")

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND"
)

job = client.load_table_from_dataframe(
    dfInputFeat,
    path_bq_data_input,
    job_config=job_config
)

job.result()


print(f"Tabla cargada: {path_bq_data_input}") 

Registros previos eliminados para período 202605.
Tabla cargada: gcp-processing-vertex-prod-us.dev_table.temp_data_input


## Componente 2

In [9]:
# Cliente de Cloud Storage
client = storage.Client(project=project_id)
bucket = client.bucket(bucket_name)


blob = bucket.blob(metadata_path)

module_code  = blob.download_as_text(encoding="utf-8")

print("Archivo PY cargado correctamente ✅")
print(type(module_code))

metadata_transformer = types.ModuleType('metadata_transformer')
metadata_transformer.__file__ = "gs://.../metadata_transformer.py"
sys.modules['metadata_transformer'] = metadata_transformer

exec(compile(module_code, metadata_transformer.__file__, "exec"),
     metadata_transformer.__dict__)

blob = bucket.blob(pipeline_path)

contenido_joblib  = blob.download_as_bytes()

pipeline = joblib.load(BytesIO(contenido_joblib))

print("Archivo JOBLIB cargado correctamente ✅")
print(type(pipeline))

Archivo PY cargado correctamente ✅
<class 'str'>
Archivo JOBLIB cargado correctamente ✅
<class 'sklearn.pipeline.Pipeline'>


In [10]:
# Cliente BigQuery
client = bigquery.Client(project=project_id)

path_bq_data_input = f"{project_id}.{dataset_temp}.{table_temp_data_input}"

dfInputFeat = client.query(
        f'''SELECT * EXCEPT(run_id, creation_user, str_process_date_local, process_datetime_local, process_datetime_utc)
        FROM `{path_bq_data_input}` where periodo = '{periodo}'
        '''
).to_dataframe()

X_escalado = pipeline.transform(dfInputFeat)

feature_order = pipeline.named_steps[
    "feature_engineering"
].get_feature_names_out()

col_orden = [col + '_transf' for col in feature_order]

dfInputFeatTransf = pd.DataFrame(
    X_escalado,
    columns = col_orden,
    index=  dfInputFeat.index,
)

user_id = client.query("SELECT SESSION_USER()").to_dataframe().iloc[0, 0]
fecha_carga = datetime.now(pytz.timezone("America/Lima"))
periodo_out = dfInputFeat.periodo[0]

dfInputFeatTransf['id'] = dfInputFeat['id']
dfInputFeatTransf['run_id'] = run_id
dfInputFeatTransf['creation_user'] = user_id
dfInputFeatTransf['str_process_date_local'] = fecha_carga.replace(tzinfo=None).strftime('%Y%m%d')
dfInputFeatTransf['process_datetime_local'] = fecha_carga.replace(tzinfo=None)
dfInputFeatTransf['process_datetime_utc'] = fecha_carga.astimezone(pytz.UTC)
dfInputFeatTransf['periodo'] = periodo_out

dfInputFeatTransf = dfInputFeatTransf[
                                    ['id','periodo'] 
                                    + col_orden 
                                    + ['run_id','creation_user','str_process_date_local','process_datetime_local','process_datetime_utc']
                    ]
dfInputFeatTransf

,id,periodo,mssubclass_transf,mszoning_transf,lotfrontage_transf,lotarea_transf,neighborhood_transf,overallqual_transf,overallcond_transf,yearbuilt_transf,yearremodadd_transf,bsmtqual_transf,bsmtfintype_principal_transf,bsmtfinsf_principal_transf,totalbsmtsf_transf,centralair_transf,firstflrsf_transf,secondflrsf_transf,grlivarea_transf,kitchenqual_transf,fireplacequ_transf,garagetype_transf,garagecars_transf,garagearea_transf,lotfrontage_na_transf,run_id,creation_user,str_process_date_local,process_datetime_local,process_datetime_utc
0,1461,202605,0.750000,0.50,0.495064,0.048246,0.363636,0.444444,0.625,0.644928,0.819672,0.333333,0.4,0.082920,0.144354,1.0,0.373438,0.000000,0.349081,0.333333,0.0,0.8,0.25,0.514810,0.0,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:24:09.299269,2026-08-08 19:24:09.299269+00:00
1,1462,202605,0.750000,0.75,0.499662,0.060609,0.363636,0.555556,0.625,0.623188,0.868852,0.333333,0.8,0.163536,0.217512,1.0,0.522632,0.000000,0.488544,0.666667,0.0,0.8,0.25,0.220028,0.0,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:24:09.299269,2026-08-08 19:24:09.299269+00:00
2,1467,202605,0.750000,0.75,0.445002,0.031223,0.590909,0.555556,0.750,0.869565,0.065574,0.666667,0.8,0.165663,0.191162,1.0,0.479870,0.000000,0.448571,0.333333,0.0,0.8,0.50,0.296192,1.0,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:24:09.299269,2026-08-08 19:24:09.299269+00:00
3,1469,202605,0.750000,0.75,0.517503,0.041487,0.590909,0.666667,0.500,0.855072,0.344262,0.666667,1.0,0.112863,0.212766,1.0,0.526034,0.000000,0.491723,0.666667,0.2,0.8,0.50,0.356841,0.0,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:24:09.299269,2026-08-08 19:24:09.299269+00:00
4,1470,202605,0.750000,0.75,0.445638,0.033186,0.363636,0.333333,0.500,0.710145,0.672131,0.333333,0.8,0.142452,0.144354,1.0,0.367478,0.000000,0.343510,0.333333,0.0,0.8,0.50,0.370240,0.0,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:24:09.299269,2026-08-08 19:24:09.299269+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1454,2772,202605,0.166667,0.75,0.445638,0.026642,0.363636,0.444444,0.500,0.652174,0.737705,0.333333,0.8,0.168852,0.167758,1.0,0.424340,0.000000,0.396663,0.333333,0.0,0.0,0.00,0.000000,0.0,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:24:09.299269,2026-08-08 19:24:09.299269+00:00
1455,2779,202605,0.166667,0.25,0.363044,0.030125,0.227273,0.333333,0.625,0.202899,0.934426,0.333333,0.0,0.000000,0.153519,0.0,0.445519,0.419855,0.624354,0.000000,0.0,0.4,0.50,0.406206,0.0,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:24:09.299269,2026-08-08 19:24:09.299269+00:00
1456,2784,202605,0.166667,0.25,0.321097,0.021968,0.227273,0.444444,0.750,0.601449,0.852459,0.333333,1.0,0.102055,0.157119,1.0,0.399547,0.000000,0.373487,0.333333,0.0,0.4,0.50,0.406206,0.0,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:24:09.299269,2026-08-08 19:24:09.299269+00:00
1457,2875,202605,0.166667,0.50,0.376033,0.023978,0.318182,0.555556,0.625,0.528986,0.934426,0.333333,0.6,0.138200,0.127660,0.0,0.338044,0.253753,0.491460,0.333333,0.0,0.8,0.25,0.310296,0.0,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:24:09.299269,2026-08-08 19:24:09.299269+00:00


In [11]:
path_bq_data_transf_input = f"{project_id}.{dataset_temp}.{table_temp_data_transf_input}"

try:
    delete_query = f"""
        DELETE FROM `{path_bq_data_transf_input}`
        WHERE periodo = '{periodo}'
    """

    delete_job = client.query(delete_query)
    delete_job.result()

    print(f"Registros previos eliminados para período {periodo}.")

except NotFound:
    print("La tabla destino no existe aún; se creará durante la carga.")

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND"
)

job = client.load_table_from_dataframe(
    dfInputFeatTransf,
    path_bq_data_transf_input,
    job_config=job_config
)

job.result()

print(f"Tabla cargada: {path_bq_data_transf_input}") 

Registros previos eliminados para período 202605.
Tabla cargada: gcp-processing-vertex-prod-us.dev_table.temp_data_transformed


## Componente 3

In [12]:
# Cliente de Cloud Storage
client = storage.Client(project=project_id)

bucket = client.bucket(bucket_name)
blob = bucket.blob(model_path)

contenido_joblib  = blob.download_as_bytes()

modelo = joblib.load(BytesIO(contenido_joblib))

print("Archivo JOBLIB cargado correctamente ✅")
print(type(modelo))

Archivo JOBLIB cargado correctamente ✅
<class 'sklearn.ensemble._gb.GradientBoostingRegressor'>


In [ ]:
# Cliente BigQuery
client = bigquery.Client(project=project_id)

path_bq_data_transf_input = f"{project_id}.{dataset_temp}.{table_temp_data_transf_input}"

dfInputFeatTransf = client.query(
        f'''SELECT * EXCEPT(run_id, creation_user, str_process_date_local, process_datetime_local, process_datetime_utc)
        FROM `{path_bq_data_transf_input}` where periodo = '{periodo}'
        '''
).to_dataframe()

dfInputFeatTransf['saleprice_predict'] = modelo.predict(dfInputFeatTransf.drop(columns = ['id','periodo']))

dfPredict = dfInputFeatTransf[['id','saleprice_predict']].copy()

user_id = client.query("SELECT SESSION_USER()").to_dataframe().iloc[0, 0]
fecha_carga = datetime.now(pytz.timezone("America/Lima"))
periodo_out = dfInputFeatTransf.periodo[0]
model_name = model_path.split('/')[-1]

dfPredict['run_id'] = run_id
dfPredict['creation_user'] = user_id
dfPredict['str_process_date_local'] = fecha_carga.replace(tzinfo=None).strftime('%Y%m%d')
dfPredict['process_datetime_local'] = fecha_carga.replace(tzinfo=None)
dfPredict['process_datetime_utc'] = fecha_carga.astimezone(pytz.UTC)
dfPredict['periodo'] = periodo_out
dfPredict['model_name'] = model_name

dfPredict =  dfPredict[
        ['id','periodo','saleprice_predict','model_name','run_id','creation_user','str_process_date_local',
         'process_datetime_local','process_datetime_utc']
].copy()
dfPredict

d:\03_PROYECTOS\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but GradientBoostingRegressor was fitted without feature names
  warnings.warn(


,id,periodo,saleprice_predict,model_name,run_id,creation_user,str_process_date_local,process_datetime_local,process_datetime_utc
0,1461,202605,128349.572714,Model-GradientBoostingRegressor.joblib,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:41:11.567608,2026-08-08 19:41:11.567608+00:00
1,1462,202605,160520.233503,Model-GradientBoostingRegressor.joblib,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:41:11.567608,2026-08-08 19:41:11.567608+00:00
2,1467,202605,168085.119359,Model-GradientBoostingRegressor.joblib,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:41:11.567608,2026-08-08 19:41:11.567608+00:00
3,1469,202605,178115.648698,Model-GradientBoostingRegressor.joblib,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:41:11.567608,2026-08-08 19:41:11.567608+00:00
4,1470,202605,124100.659206,Model-GradientBoostingRegressor.joblib,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:41:11.567608,2026-08-08 19:41:11.567608+00:00
...,...,...,...,...,...,...,...,...,...
1454,2772,202605,122625.935331,Model-GradientBoostingRegressor.joblib,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:41:11.567608,2026-08-08 19:41:11.567608+00:00
1455,2779,202605,107615.117360,Model-GradientBoostingRegressor.joblib,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:41:11.567608,2026-08-08 19:41:11.567608+00:00
1456,2784,202605,125906.519672,Model-GradientBoostingRegressor.joblib,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:41:11.567608,2026-08-08 19:41:11.567608+00:00
1457,2875,202605,125376.275248,Model-GradientBoostingRegressor.joblib,house-price-20260808T192339Z-8625ec275def,sa-processing-crun-vertex-prod@gcp-processing-...,20260808,2026-08-08 14:41:11.567608,2026-08-08 19:41:11.567608+00:00


In [23]:
path_bq_data_predict = f"{project_id}.{dataset_temp}.{table_temp_data_predict}"

try:
    delete_query = f"""
        DELETE FROM `{path_bq_data_predict}`
        WHERE periodo = '{periodo}'
    """

    delete_job = client.query(delete_query)
    delete_job.result()

    print(f"Registros previos eliminados para período {periodo}.")

except NotFound:
    print("La tabla destino no existe aún; se creará durante la carga.")

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND"
)

job = client.load_table_from_dataframe(
    dfPredict,
    path_bq_data_predict,
    job_config=job_config
)

job.result()

print(f"Tabla cargada: {path_bq_data_predict}") 

Registros previos eliminados para período 202605.
Tabla cargada: gcp-processing-vertex-prod-us.dev_table.temp_data_predict


In [24]:
table_id_auditoria = f"{project_id}.{dataset_id_features}.model_execution_audit"

user_id = client.query("SELECT SESSION_USER()").to_dataframe().iloc[0, 0]
fecha_carga = datetime.now(pytz.timezone("America/Lima"))

df_auditoria = pd.DataFrame([{
    "periodo": periodo_out,
    "run_id" : run_id,
    "model_name": model_name,
    "model_path_gcs": f"gs://{bucket_name}/{model_path}",
    "pipeline_path_gcs": (
        f"gs://{bucket_name}/{model_ruta}/"
    ),
    "rows_processed": len(dfPredict),
    "str_process_date_local": fecha_carga.replace(tzinfo=None).strftime('%Y%m%d'),
    "process_datetime_local": fecha_carga.replace(tzinfo=None),
    "process_datetime_utc": fecha_carga.astimezone(pytz.UTC)
}])

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND"
)

job = client.load_table_from_dataframe(
    df_auditoria,
    table_id_auditoria,
    job_config=job_config
)

job.result()

print(f"Auditoría registrada en: {table_id_auditoria}")

Auditoría registrada en: gcp-processing-vertex-prod-us.dev_table.model_execution_audit


## Componente 4

In [31]:
# Cliente BigQuery
client = bigquery.Client(project=project_id)

query = f"""
CALL `gcp-processing-vertex-prod-us.dev_table.sp_table_out_model` ('{periodo}')
"""

job = client.query(query)
dfOut = job.result().to_dataframe()
fecha_carga = datetime.now(pytz.timezone("America/Lima"))
dfOut['str_process_date_local'] = fecha_carga.replace(tzinfo=None).strftime('%Y%m%d')
dfOut['process_datetime_local'] = fecha_carga.replace(tzinfo=None)
dfOut['process_datetime_utc'] = fecha_carga.astimezone(pytz.UTC)
dfOut

,id,periodo,mssubclass,mszoning,lotfrontage,lotarea,neighborhood,overallqual,overallcond,yearbuilt,yearremodadd,bsmtqual,bsmtfintype_principal,bsmtfinsf_principal,totalbsmtsf,centralair,firstflrsf,secondflrsf,grlivarea,kitchenqual,fireplacequ,garagetype,garagecars,garagearea,yrsold,saleprice,str_process_date_local,process_datetime_local,process_datetime_utc
0,1823,202605,30,C (all),72.0,9392,IDOTRR,3,3,1900,1950,Fa,Unf,0.0,245.0,N,797.0,0.0,797.0,TA,None,None,0.0,0.0,2009,32647.933592,20260808,2026-08-08 16:02:36.515123,2026-08-08 21:02:36.515123+00:00
1,2894,202605,50,C (all),60.0,8520,IDOTRR,3,5,1916,1950,Fa,Unf,0.0,216.0,N,576.0,360.0,936.0,TA,None,None,0.0,0.0,2006,41599.909737,20260808,2026-08-08 16:02:36.515123,2026-08-08 21:02:36.515123+00:00
2,2792,202605,50,C (all),63.0,4761,IDOTRR,3,3,1918,1950,TA,Unf,0.0,1020.0,N,1020.0,0.0,1020.0,Fa,None,None,0.0,0.0,2006,43793.940362,20260808,2026-08-08 16:02:36.515123,2026-08-08 21:02:36.515123+00:00
3,2872,202605,30,RL,60.0,8088,Edwards,2,3,1922,1955,TA,Unf,0.0,498.0,N,498.0,0.0,498.0,TA,None,Detchd,1.0,216.0,2006,45392.580293,20260808,2026-08-08 16:02:36.515123,2026-08-08 21:02:36.515123+00:00
4,2892,202605,30,C (all),69.0,12366,IDOTRR,3,5,1945,1950,None,None,0.0,0.0,N,729.0,0.0,729.0,TA,None,None,0.0,0.0,2006,47317.936969,20260808,2026-08-08 16:02:36.515123,2026-08-08 21:02:36.515123+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1454,2683,202605,60,RL,114.0,17242,NoRidge,9,5,1993,1994,Ex,Rec,292.0,1733.0,Y,1933.0,1567.0,3500.0,Ex,TA,Attchd,3.0,959.0,2006,529242.416377,20260808,2026-08-08 16:02:36.515123,2026-08-08 21:02:36.515123+00:00
1455,1672,202605,20,RL,110.0,15274,NridgHt,9,5,2003,2003,Ex,GLQ,1972.0,2452.0,Y,2452.0,0.0,2452.0,Ex,Gd,Attchd,3.0,886.0,2009,535928.563818,20260808,2026-08-08 16:02:36.515123,2026-08-08 21:02:36.515123+00:00
1456,2293,202605,20,RL,107.0,13891,NridgHt,9,5,2007,2007,Ex,GLQ,1812.0,2552.0,Y,2552.0,0.0,2552.0,Ex,Ex,Attchd,3.0,932.0,2007,543448.081671,20260808,2026-08-08 16:02:36.515123,2026-08-08 21:02:36.515123+00:00
1457,1678,202605,20,RL,100.0,14836,NridgHt,10,5,2004,2005,Ex,GLQ,2146.0,2492.0,Y,2492.0,0.0,2492.0,Ex,Ex,Attchd,3.0,949.0,2009,550270.983235,20260808,2026-08-08 16:02:36.515123,2026-08-08 21:02:36.515123+00:00


In [32]:
table_id_out_hist = f'{project_id_out}.{dataset_id_out}.{table_out_hist}'

try:

    delete_query = f"""
        DELETE FROM `{table_id_out_hist}`
        WHERE periodo = '{periodo}'
    """

    delete_job = client.query(delete_query)
    delete_job.result()

    print(f"Registros previos eliminados para período {periodo}.")

except NotFound:
    print("La tabla destino no existe aún; se creará durante la carga.")

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND"
)

job = client.load_table_from_dataframe(
    dfOut,
    table_id_out_hist,
    job_config=job_config
)

job.result()

print(f"Subida a PRD registrada en: {table_id_out_hist}")

Registros previos eliminados para período 202605.
Subida a PRD registrada en: gcp-output-bigquery-prod-us-ea.prod_table.ba_aux_modelo_universo


In [33]:
table_id_out = f'{project_id_out}.{dataset_id_out}.{table_out}'

try:

    delete_query = f"""
        DELETE FROM `{table_id_out}`
        WHERE periodo = '{periodo}'
    """

    delete_job = client.query(delete_query)
    delete_job.result()

    print(f"Registros previos eliminados para período {periodo}.")

except NotFound:
    print("La tabla destino no existe aún; se creará durante la carga.")

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

job = client.load_table_from_dataframe(
    dfOut,
    table_id_out,
    job_config=job_config
)

job.result()

print(f"Subida a PRD registrada en: {table_id_out}")

Registros previos eliminados para período 202605.
Subida a PRD registrada en: gcp-output-bigquery-prod-us-ea.prod_table.ba_modelo_propension_inscripcion
